In [10]:
import pandas as pd
import numpy as np
import re

# 1. 读取原始数据
df = pd.read_csv(r"C:\Users\TARUMT\Downloads\data.csv")

rows = []

# 2. 遍历每场比赛，拆解为主队视角和客队视角
for idx, row in df.iterrows():
    # 处理带点球大战的复杂比分格式，如 '(1) 1–1 (3)' -> 提取 90分钟内比分 '1-1'
    clean_score = re.sub(r'\(.*?\)', '', str(row['score'])).replace('–', '-').strip()
    parts = clean_score.split('-')
    h_score, a_score = int(parts[0].strip()), int(parts[1].strip())
    
    # 计算传球成功率 (Pass Accuracy)
    h_pass_acc = round((row['home_completed_passes'] / row['home_attempted_pases']) * 100, 2) if row['home_attempted_pases'] > 0 else 0
    a_pass_acc = round((row['away_completed_passes'] / row['away_attempted_pases']) * 100, 2) if row['away_attempted_pases'] > 0 else 0
    
    # 估算 PPDA (对手传球数 / (抢断 + 拦截))
    h_ppda = round(row['away_completed_passes'] / max(row['home_tackles'] + row['home_interceptions'], 1), 2)
    a_ppda = round(row['home_completed_passes'] / max(row['away_tackles'] + row['away_interceptions'], 1), 2)
    
    # 空中对抗胜率 (%)
    total_aerials = max(row['home_aerials_won'] + row['away_aerials_won'], 1)
    h_aerial = round((row['home_aerials_won'] / total_aerials) * 100, 2)
    a_aerial = round((row['away_aerials_won'] / total_aerials) * 100, 2)

    # ------------------ 主队视角 ------------------
    h_outcome = 'Win' if h_score > a_score else ('Loss' if h_score < a_score else 'Draw')
    rows.append({
        'match_id': row['match'],
        'team': row['home_team'],
        'opponent': row['away_team'],
        'xg': float(row['home_xg']),
        'possession': float(row['home_possession']),
        'shots_on_target': float(row['home_sot']),
        'shots_total': float(row['home_total_shots']),
        'passes_completed': float(row['home_completed_passes']),
        'pass_accuracy': h_pass_acc,
        'ppda': h_ppda,
        'tackles_successful': float(row['home_tackles']),
        'interceptions': float(row['home_interceptions']),
        'clearances': float(row['home_clearances']),
        'fouls_committed': float(row['home_fouls']),
        'yellow_cards': 0,  # 原始数据集未包含，设默认值
        'corners': float(row['home_corners']),
        'crosses_completed': float(row['home_crosses']),
        'aerial_duels_won_pct': h_aerial,
        'errors_leading_to_shot': 0, # 原始数据集未包含，设默认值
        'outcome': h_outcome
    })

    # ------------------ 客队视角 ------------------
    a_outcome = 'Win' if a_score > h_score else ('Loss' if a_score < h_score else 'Draw')
    rows.append({
        'match_id': row['match'],
        'team': row['away_team'],
        'opponent': row['home_team'],
        'xg': float(row['away_xg']),
        'possession': float(row['away_possession']),
        'shots_on_target': float(row['away_sot']),
        'shots_total': float(row['away_total_shots']),
        'passes_completed': float(row['away_completed_passes']),
        'pass_accuracy': a_pass_acc,
        'ppda': a_ppda,
        'tackles_successful': float(row['away_tackles']),
        'interceptions': float(row['away_interceptions']),
        'clearances': float(row['away_clearances']),
        'fouls_committed': float(row['away_fouls']),
        'yellow_cards': 0,
        'corners': float(row['away_corners']),
        'crosses_completed': float(row['away_crosses']),
        'aerial_duels_won_pct': a_aerial,
        'errors_leading_to_shot': 0,
        'outcome': a_outcome
    })

# 3. 转成标准的 Clean DataFrame
clean_df = pd.DataFrame(rows)

# 保存清洗后的文件
clean_df.to_csv('clean_world_cup_2022.csv', index=False)

print("✅ 数据处理完毕！成功转换成 128 条队伍战术记录。")
print(clean_df[['team', 'opponent', 'xg', 'possession', 'outcome']].head())

✅ 数据处理完毕！成功转换成 128 条队伍战术记录。
      team     opponent   xg  possession outcome
0    Qatar      Ecuador  0.3        47.0    Loss
1  Ecuador        Qatar  1.2        53.0     Win
2  England      IR Iran  2.1        77.0     Win
3  IR Iran      England  1.4        23.0    Loss
4  Senegal  Netherlands  0.9        46.0    Loss


In [11]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 读取刚刚清洗好的数据
df = pd.read_csv('clean_world_cup_2022.csv')

# 16 个战术特征列表
tactical_features = [
    'xg', 'possession', 'shots_on_target', 'shots_total',
    'passes_completed', 'pass_accuracy', 'ppda', 'tackles_successful',
    'interceptions', 'clearances', 'fouls_committed', 'yellow_cards',
    'corners', 'crosses_completed', 'aerial_duels_won_pct', 'errors_leading_to_shot'
]

X = df[tactical_features]
y = df['outcome']

# 使用 Stratified K-Fold 5折交叉验证
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy')

print(f"🎯 5-Fold Cross-Validation 平均准确率: {scores.mean():.4f}")

# 用全量数据训练并保存 .pkl 文件给 Streamlit 界面使用
rf_model.fit(X, y)
joblib.dump(rf_model, 'world_cup_rf_model.pkl')
print("✅ 模型已顺利导出为 world_cup_rf_model.pkl！")

🎯 5-Fold Cross-Validation 平均准确率: 0.4440
✅ 模型已顺利导出为 world_cup_rf_model.pkl！
